# Notebook 1 — Theory of Max-Cut QAOA

**Goal**: Derive, from first principles, the QAOA algorithm for Max-Cut. By the end of this notebook you should be able to:

1. Write the Max-Cut objective as an Ising Hamiltonian on qubits.
2. Explain the role of the cost Hamiltonian $H_C$ and the mixer Hamiltonian $H_M$.
3. Write the QAOA ansatz $|\psi(\gamma,\beta)\rangle$ at depth $p$.
4. Derive the gradient structure and explain why deeper QAOA gets closer to optimal.

**References**: Farhi, Goldstone, Gutmann (2014), arXiv:1411.4028.

## 1. The Max-Cut problem

Given an undirected graph $G=(V,E)$ with $|V|=n$ nodes and $|E|=m$ edges, the **Max-Cut problem** is to find a partition of $V$ into two sets $A$ and $B$ that maximizes the number of edges crossing the cut:

$$\text{Maximize} \quad C(z) = \sum_{(i,j)\in E} [z_i \oplus z_j]$$

where $z_i \in \{0,1\}$ is the side of node $i$ and $\oplus$ is XOR (the edge is cut iff $z_i \ne z_j$).

**Max-Cut is NP-hard.** The best known polynomial-time classical approximation is Goemans-Williamson (1995), with a guaranteed ratio $\ge 0.878$.

In [ ]:
import sys; sys.path.insert(0, '../src')
from max_cut import MaxCut

# A simple 5-node graph: triangle (0,1,2) + two appendages (3 from 1, 4 from 0)
mc = MaxCut.from_edges(5, [(0,1),(1,2),(2,0),(1,3),(3,4),(4,0)], name='5-node test')
print(mc.summary())
mc.draw(title='Our 5-node graph')

## 2. From Max-Cut to an Ising Hamiltonian

Substitute $z_i = (1 - s_i)/2$ where $s_i \in \{-1,+1\}$ is a spin. Then $[z_i \oplus z_j] = (1 - s_i s_j)/2$ (the edge is cut iff the spins differ). The objective becomes

$$C(s) = \frac{1}{2}\sum_{(i,j)\in E}(1 - s_i s_j) = \frac{m}{2} - \frac{1}{2}\sum_{(i,j)\in E} s_i s_j.$$

Promote each spin to a Pauli-$Z$ operator $s_i \to Z_i$ (eigenvalues $\pm 1$). The **cost Hamiltonian** is

$$H_C = -\frac{1}{2}\sum_{(i,j)\in E}(1 - Z_i Z_j) = -\frac{m}{2} + \frac{1}{2}\sum_{(i,j)\in E} Z_i Z_j.$$

Maximizing $C$ corresponds to minimizing $\langle H_C\rangle$ (the constant offset $-m/2$ is irrelevant).

In [ ]:
# Inspect the cost Hamiltonian
H_C = mc.cost_hamiltonian()
print('Cost Hamiltonian H_C =')
print(H_C)
print(f'\n({len(H_C)} terms — one per edge)')

## 3. The mixer Hamiltonian

QAOA also needs a **mixer** Hamiltonian $H_M$ that does not commute with $H_C$, used to rotate the state out of local optima. The standard choice (Farhi et al.) is the transverse-field mixer:

$$H_M = \sum_{i=1}^n X_i.$$

$H_M$ flips qubits and provides the variational freedom needed for the algorithm to converge to the ground state of $H_C$.

In [ ]:
H_M = mc.mixer_hamiltonian()
print('Mixer Hamiltonian H_M =')
print(H_M)

## 4. The QAOA ansatz

QAOA prepares a parameterized state by alternating cost and mixer evolutions:

$$|\psi(\gamma,\beta)\rangle = e^{-i\beta_p H_M} e^{-i\gamma_p H_C} \cdots e^{-i\beta_1 H_M} e^{-i\gamma_1 H_C} |+\rangle^{\otimes n}$$

where $|+\rangle^{\otimes n} = H^{\otimes n} |0\rangle^{\otimes n}$ is the uniform superposition, and $p$ is the **depth** of the ansatz. The variational parameters are $\gamma = (\gamma_1, \dots, \gamma_p)$ and $\beta = (\beta_1, \dots, \beta_p)$.

**Theorem (Farhi et al. 2014)**: As $p \to \infty$, QAOA finds the exact optimum.

**At small $p$**, the approximation ratio $\langle C \rangle / C^*$ is bounded and known analytically for some graph families (e.g., for 3-regular graphs at $p=1$, the ratio is at least $0.6924$).

In [ ]:
from qaoa import build_qaoa_circuit
from qiskit.visualization import circuit_drawer

# Build the circuit at p=1 with sample parameters
qc = build_qaoa_circuit(mc, gammas=[0.5], betas=[0.3])
print('QAOA circuit at p=1:')
print(qc.draw(output='text'))

### Why each gate is what it is

* **Hadamard** on each qubit: prepares $|+\rangle^{\otimes n}$, the uniform superposition over all $2^n$ bitstrings.
* **`RZZ(2γ)`** for each edge $(i,j)$: implements $e^{-i\gamma Z_i Z_j}$, the cost evolution on that edge. The factor of 2 in `RZZ(2γ)` absorbs the $1/2$ from $H_C = -(1/2) Z_i Z_j$.
* **`Rx(2β)`** on each qubit: implements $e^{-i\beta X_i}$, the mixer evolution. Again the factor of 2 absorbs the convention.
* **Measurement** in the computational basis: each shot returns a candidate bitstring $z$.

For $p=1$ the circuit has $n + m + n = 2n + m$ gates. For our 5-node 6-edge graph that's 16 gates — tiny.

## 5. The classical outer loop

QAOA is a **variational quantum eigensolver (VQE)**:

1. **Prepare** $|\psi(\gamma,\beta)\rangle$ on the quantum computer.
2. **Measure** $\langle H_C \rangle = \langle \psi(\gamma,\beta) | H_C | \psi(\gamma,\beta) \rangle$ by sampling.
3. **Update** $(\gamma,\beta)$ with a classical optimizer (e.g., COBYLA, SPSA) to *minimize* $\langle H_C \rangle$.
4. Repeat until convergence. The final measurement gives samples from the (approximate) ground state.

Because $\langle H_C \rangle = -\langle C \rangle$ (the mean cut value is $-\langle H_C \rangle$), minimizing $\langle H_C \rangle$ is equivalent to maximizing the mean cut.

## 6. Why does QAOA work? Intuition

Think of $H_C$ as defining an energy landscape over the $2^n$ bitstrings, with the ground state (minimum energy) being the optimal cut. The cost evolution $e^{-i\gamma H_C}$ tilts the amplitudes towards low-energy (high-cut) states. The mixer $e^{-i\beta H_M}$ rotates the state in directions that don't commute with $H_C$, allowing the algorithm to escape local minima.

**At $p=1$**: we make one tilt + one rotation. The state moves towards the optimum but typically gets stuck in a nearby local min.

**At $p=2, 3, \dots$**: each extra layer gives another (gentler) tilt + rotation. More layers → closer to the true ground state. This is the variational trade-off: more parameters to optimize, but better final quality.

**At $p \to \infty$**: the construction approaches the **adiabatic theorem** — if we slowly interpolate from $H_M$ to $H_C$ over a long time, the state tracks the ground state of the instantaneous Hamiltonian. This is the deep connection between QAOA and adiabatic quantum computing.

## 7. What's next

Move to **Notebook 2: QAOA Simulator** to actually run the algorithm on the Qiskit simulator and watch the approximation ratio climb with $p$.